# SKENARIO 3 — CoT vs non-CoT (fine-tuned) — **Colab A100**

Bandingkan **model fine-tune CoT** vs **non-CoT** di test set yang SAMA.
Base + hyperparam identik, beda **cuma data training (cot vs nocot)** -> selisih = efek CoT.

**Sampling** (N=5 kandidat/soal, temp 0.7), ekstrak `\boxed{}`, cocokkan ke gold. Butuh **GPU**.
Metrik: **pass@1, pass@2, pass@3, maj@3, maj@5** + delta (CoT − nonCoT) per metrik/set.

Runtime: **GPU A100** (Runtime -> Change runtime type -> A100).
**Checkpoint/resume**: generasi tiap soal dicache per (arm × dataset); run ulang nge-skip yang sudah selesai.

## 0. Install + clone repo

In [ ]:
!pip install -q -U peft transformers accelerate
# peft>=0.18 manggil dispatcher torchao; copot saja biar peft skip jalurnya (no-op kalau gak ada).
!pip uninstall -y -q torchao

In [ ]:
import os, sys, subprocess
from pathlib import Path
REPO = '/content/FP_NLP'
URL  = 'https://github.com/henray404/FP_NLP.git'
if os.path.exists(REPO):
    subprocess.run(['git','-C',REPO,'fetch','-q','origin','main'], check=True)
    subprocess.run(['git','-C',REPO,'reset','--hard','origin/main'], check=True)
else:
    subprocess.run(['git','clone','-q',URL,REPO], check=True)
sys.path.insert(0, REPO); os.chdir(REPO)
print('repo ready:', REPO, '| cwd:', os.getcwd())

## 1. Adapter — Drive / zip / upload (robust)

Cari folder adapter langsung (mis. `MyDrive/adapters/adapter_cot_1.5b/`). Kalau ketemu -> dipakai,
**tanpa unzip**. Kalau belum ada, cari `adapters_s3.zip` (Drive / `/content` / cwd) lalu extract;
kalau tetap tak ada, munculin dialog upload.

Catatan: cell clone tadi `os.chdir(REPO)`, jadi `files.upload()` mendarat di **cwd** (`/content/FP_NLP`),
bukan `/content/`. Cell ini sudah handle itu.

In [ ]:
import glob, os, zipfile
from pathlib import Path

try:
    from google.colab import drive
    if not os.path.exists('/content/drive/MyDrive'):
        drive.mount('/content/drive')
except Exception as e:
    print('drive skip:', e)

ROOTS = ['/content/drive/MyDrive', '/content/adapters', '/content', os.getcwd()]
def find_adapter(keyword):
    for root in ROOTS:
        hits = glob.glob(f'{root}/**/*{keyword}*/adapter_config.json', recursive=True)
        if hits: return str(Path(hits[0]).parent)
    return None

if not (find_adapter('adapter_cot') and find_adapter('adapter_nocot')):
    zc = []
    for root in ['/content/drive/MyDrive', '/content', os.getcwd()]:
        zc += glob.glob(f'{root}/**/adapters_s3.zip', recursive=True)
    if not zc:
        from google.colab import files
        print('Upload adapters_s3.zip ...')
        up = files.upload()
        zc = [os.path.join(os.getcwd(), list(up)[0])]   # files.upload() simpan ke cwd
    os.makedirs('/content/adapters', exist_ok=True)
    with zipfile.ZipFile(zc[0]) as z:
        z.extractall('/content/adapters')
    print('extracted', zc[0], '-> /content/adapters')

print('CoT  :', find_adapter('adapter_cot'))
print('nonCoT:', find_adapter('adapter_nocot'))

## 2. (opsional) Checkpoint lintas-sesi di Drive

`/content` kehapus kalau sesi Colab mati. Default checkpoint ke `data/eval/ckpt_s3` (lokal, cukup utk
resume dalam 1 sesi). Buat selamat **saat sesi putus**, arahkan `CKPT_DIR` ke Drive (Drive sudah
ter-mount di cell 1).

In [ ]:
CKPT_DIR = 'data/eval/ckpt_s3'   # default lokal /content
# CKPT_DIR = '/content/drive/MyDrive/fpnlp_ckpt/s3'   # uncomment: resume lintas-sesi
os.makedirs(CKPT_DIR, exist_ok=True)
print('checkpoint dir:', CKPT_DIR)

## 3. Config — base, 2 adapter, test set

In [ ]:
BASE = 'Qwen/Qwen2.5-1.5B'
ADAPTER_COT   = find_adapter('adapter_cot')
ADAPTER_NOCOT = find_adapter('adapter_nocot')
assert ADAPTER_COT and ADAPTER_NOCOT, f'adapter kurang: cot={ADAPTER_COT} nocot={ADAPTER_NOCOT}'
print('CoT  :', ADAPTER_COT)
print('nonCoT:', ADAPTER_NOCOT)

# test set sudah ke-commit di repo -> langsung dari data/sft/test/
SET_PATHS = {'numglue': 'data/sft/test/numglue_test.jsonl', 'easy': 'data/sft/test/easy_test.jsonl'}
for k, v in SET_PATHS.items():
    assert os.path.exists(v), f'missing {v}'
print('test sets:', SET_PATHS)

## 4. Eval CoT vs non-CoT + tabel

`batch_size=48` (A100 40GB; tiap soal mekar jadi N=5 sekuens -> ~240 sekuens paralel).
Kalau OOM turunkan; kalau VRAM lega naikkan ke 64.
`ckpt_dir=CKPT_DIR` -> 4 file cache (`CoT__numglue.jsonl`, `CoT__easy.jsonl`, `nonCoT__numglue.jsonl`,
`nonCoT__easy.jsonl`). Run ulang cell ini = lanjut dari yang sudah ada.

In [ ]:
from src.eval.scenario_eval import load_sets
from src.eval.sample_eval import eval_specs_sampling
from src.eval.sampling_metrics import render_tables
import json

sets = load_sets(SET_PATHS)
specs = {
    'CoT':    {'model': BASE, 'adapter': ADAPTER_COT},
    'nonCoT': {'model': BASE, 'adapter': ADAPTER_NOCOT},
}
METRICS = ['pass@1', 'pass@2', 'pass@3', 'maj@3', 'maj@5']

results = eval_specs_sampling(specs, sets, n_samples=5, ks_pass=(1, 2, 3), ks_maj=(3, 5),
                             temperature=0.7, top_p=0.95, max_new_tokens=2048, batch_size=48,
                             ckpt_dir=CKPT_DIR)

print('\n=== SKENARIO 3: CoT vs non-CoT (sampling) ===')
print(render_tables(results, METRICS))

print('\ndelta (CoT - nonCoT):')
for m in METRICS:
    for s in sorted(sets):
        c = results['CoT'][s][m]; n = results['nonCoT'][s][m]
        print(f'  {m:7} {s:10} CoT={c:.3f} nonCoT={n:.3f} delta={c-n:+.3f}')

Path('data/eval').mkdir(parents=True, exist_ok=True)
Path('data/eval/s3_cot_vs_noncot.json').write_text(json.dumps(results, ensure_ascii=False, indent=2), encoding='utf-8')
print('\nsummary -> data/eval/s3_cot_vs_noncot.json')